In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
from datetime import datetime, timedelta
import os, shutil, time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara-Streaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/06 12:45:51 WARN Utils: Your hostname, xcel resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/06 12:45:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 12:45:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**A. Menyiapkan Streaming Source**

Skema wajib didefinisikan eksplisit *(StructType)* karena streaming tidak dapat melakukan *inferSchema*. Folder *tugas10_stream* berperan sebagai sumber data streaming (setiap berkas CSV baru dianggap data yang "mengalir masuk").

In [2]:
# Membersihkan sisa eksekusi sebelumnya agar Run All selalu mulai dari kondisi bersih
folder_stream = "tugas10_stream"
for f in [folder_stream, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(f, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

skema_pesanan = StructType([
    StructField("pesanan_id",   StringType(),    True),
    StructField("nama_cabang",  StringType(),    True),
    StructField("nama_menu",    StringType(),    True),
    StructField("qty",          IntegerType(),   True),
    StructField("harga_satuan", IntegerType(),   True),
    StructField("waktu_pesan",  TimestampType(), True),
])

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("Skema pesanan:")
df_stream.printSchema()
print("df_stream.isStreaming:", df_stream.isStreaming)

Skema pesanan:
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)

df_stream.isStreaming: True


**B. Agregasi per Cabang**

Kolom turunan *total_harga = qty x harga_satuan* ditambahkan, lalu dikelompokkan per *nama_cabang* (total *total_harga* dan jumlah pesanan). Hasil ditulis ke *memory* sink dengan nama *agregasi_cabang*.

In [3]:
df_stream_hitung = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_stream_hitung.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang dimulai. Aktif:", query_cabang.isActive)

Query agregasi_cabang dimulai. Aktif: True


**C. Agregasi Berbasis Window Waktu (2 Menit)**

Query kedua membaca ulang folder yang sama lewat *readStream* baru, lalu mengelompokkan total *total_harga* per window 2 menit dari kolom *waktu_pesan*.

In [4]:
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_harga")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi_waktu dimulai. Aktif:", query_waktu.isActive)

Query agregasi_waktu dimulai. Aktif: True


**D. Menguji dengan Data Simulasi (4 Batch)**

Empat berkas CSV dikirim bergantian dengan jeda *time.sleep()*, dengan kombinasi *nama_cabang* yang bervariasi. Setelah tiap batch, *agregasi_cabang* ditampilkan, dan hasilnya dibandingkan dengan perhitungan pandas atas seluruh data yang sudah terkirim sebagai bukti akumulasi benar.

In [5]:
semua_batch = []   # menyimpan seluruh data terkirim untuk verifikasi

def kirim_batch(nama_berkas, baris):
    """Menulis satu berkas CSV baru ke folder streaming."""
    df = pd.DataFrame(baris, columns=[
        "pesanan_id", "nama_cabang", "nama_menu", "qty", "harga_satuan", "waktu_pesan"
    ])
    df.to_csv(f"{folder_stream}/{nama_berkas}", index=False)
    semua_batch.append(df)

def tampilkan_dan_verifikasi(judul):
    print(f"=== {judul} ===")
    spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()
    # Hitung ulang manual dengan pandas dari seluruh data yang sudah dikirim
    gabungan = pd.concat(semua_batch)
    gabungan["total_harga"] = gabungan["qty"] * gabungan["harga_satuan"]
    harapan = gabungan.groupby("nama_cabang").agg(
        total_harga=("total_harga", "sum"),
        jumlah_pesanan=("pesanan_id", "count")
    ).sort_index()
    print("Hasil yang diharapkan (pandas):")
    print(harapan, "\n")
    hasil = spark.sql("SELECT * FROM agregasi_cabang").toPandas().set_index("nama_cabang").sort_index()
    cocok = (hasil["total_harga"].astype(int).equals(harapan["total_harga"].astype(int))
             and hasil["jumlah_pesanan"].astype(int).equals(harapan["jumlah_pesanan"].astype(int)))
    print("Cocok dengan perhitungan manual:", cocok, "\n")

sekarang = datetime.now()
fmt = lambda dt: dt.strftime("%Y-%m-%d %H:%M:%S")

In [6]:
# Batch 1
kirim_batch("batch_1.csv", [
    ["P001", "Magelang",   "Es Kopi Gula Aren", 3, 20000, fmt(sekarang)],
    ["P002", "Magelang",   "Kopi Tubruk",       2, 12000, fmt(sekarang)],
    ["P003", "Purworejo",  "Cold Brew",         1, 25000, fmt(sekarang)],
    ["P004", "Temanggung", "Kopi Jahe",         4, 15000, fmt(sekarang)],
])
print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(10)
tampilkan_dan_verifikasi("Setelah Batch 1")

Batch 1 dikirim. Menunggu Spark memproses...


=== Setelah Batch 1 ===
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|      84000|             2|
|  Purworejo|      25000|             1|
| Temanggung|      60000|             1|
+-----------+-----------+--------------+

Hasil yang diharapkan (pandas):
             total_harga  jumlah_pesanan
nama_cabang                             
Magelang           84000               2
Purworejo          25000               1
Temanggung         60000               1 

Cocok dengan perhitungan manual: True 



In [7]:
# Batch 2: cabang Salatiga muncul, Magelang & Purworejo bertambah
kirim_batch("batch_2.csv", [
    ["P005", "Salatiga",  "Flat White",        2, 24000, fmt(sekarang + timedelta(seconds=40))],
    ["P006", "Magelang",  "Es Kopi Pandan",    5, 22000, fmt(sekarang + timedelta(seconds=40))],
    ["P007", "Purworejo", "Kopi Tubruk",       3, 12000, fmt(sekarang + timedelta(seconds=40))],
])
print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(10)
tampilkan_dan_verifikasi("Setelah Batch 2")

Batch 2 dikirim. Menunggu Spark memproses...


=== Setelah Batch 2 ===
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     194000|             3|
|  Purworejo|      61000|             2|
|   Salatiga|      48000|             1|
| Temanggung|      60000|             1|
+-----------+-----------+--------------+

Hasil yang diharapkan (pandas):
             total_harga  jumlah_pesanan
nama_cabang                             
Magelang          194000               3
Purworejo          61000               2
Salatiga           48000               1
Temanggung         60000               1 

Cocok dengan perhitungan manual: True 



In [8]:
# Batch 3: cabang Wonosobo muncul, waktu pesan digeser ke window 2 menit berikutnya
kirim_batch("batch_3.csv", [
    ["P008", "Wonosobo",   "Kopi Jahe",         6, 15000, fmt(sekarang + timedelta(minutes=2, seconds=30))],
    ["P009", "Temanggung", "Es Kopi Gula Aren", 2, 20000, fmt(sekarang + timedelta(minutes=2, seconds=30))],
    ["P010", "Salatiga",   "Cold Brew",         3, 25000, fmt(sekarang + timedelta(minutes=2, seconds=30))],
    ["P011", "Magelang",   "Flat White",        1, 24000, fmt(sekarang + timedelta(minutes=2, seconds=30))],
    ["P012", "Wonosobo",   "Kopi Tubruk",       4, 12000, fmt(sekarang + timedelta(minutes=2, seconds=30))],
])
print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(10)
tampilkan_dan_verifikasi("Setelah Batch 3")

Batch 3 dikirim. Menunggu Spark memproses...


=== Setelah Batch 3 ===
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     218000|             4|
|  Purworejo|      61000|             2|
|   Salatiga|     123000|             2|
| Temanggung|     100000|             2|
|   Wonosobo|     138000|             2|
+-----------+-----------+--------------+

Hasil yang diharapkan (pandas):
             total_harga  jumlah_pesanan
nama_cabang                             
Magelang          218000               4
Purworejo          61000               2
Salatiga          123000               2
Temanggung        100000               2
Wonosobo          138000               2 

Cocok dengan perhitungan manual: True 



In [9]:
# Batch 4: window waktu yang lebih jauh lagi
kirim_batch("batch_4.csv", [
    ["P013", "Purworejo", "Es Kopi Pandan",    2, 22000, fmt(sekarang + timedelta(minutes=5))],
    ["P014", "Magelang",  "Kopi Jahe",         3, 15000, fmt(sekarang + timedelta(minutes=5))],
    ["P015", "Salatiga",  "Es Kopi Gula Aren", 4, 20000, fmt(sekarang + timedelta(minutes=5))],
])
print("Batch 4 dikirim. Menunggu Spark memproses...")
time.sleep(10)
tampilkan_dan_verifikasi("Setelah Batch 4")

Batch 4 dikirim. Menunggu Spark memproses...


=== Setelah Batch 4 ===
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     263000|             5|
|  Purworejo|     105000|             3|
|   Salatiga|     203000|             3|
| Temanggung|     100000|             2|
|   Wonosobo|     138000|             2|
+-----------+-----------+--------------+

Hasil yang diharapkan (pandas):
             total_harga  jumlah_pesanan
nama_cabang                             
Magelang          263000               5
Purworejo         105000               3
Salatiga          203000               3
Temanggung        100000               2
Wonosobo          138000               2 

Cocok dengan perhitungan manual: True 



In [10]:
# Hasil query window waktu 2 menit (Bagian C) setelah keempat batch masuk
spark.sql("""
    SELECT window.start AS awal_window, window.end AS akhir_window, total_harga
    FROM agregasi_waktu
    ORDER BY window.start
""").show(truncate=False)

+-------------------+-------------------+-----------+
|awal_window        |akhir_window       |total_harga|
+-------------------+-------------------+-----------+
|2026-11-06 12:46:00|2026-11-06 12:48:00|363000     |
|2026-11-06 12:48:00|2026-11-06 12:50:00|277000     |
|2026-11-06 12:50:00|2026-11-06 12:52:00|169000     |
+-------------------+-------------------+-----------+



**E. Sink Berbasis Berkas (Tantangan)**

Query ketiga menulis data mentah (tanpa agregasi) dari *df_stream* ke folder *tugas10_output* dalam format Parquet dengan mode *append*. Karena query ini baru dimulai setelah Batch 1–4 sudah ada di folder, Spark akan ikut memproses berkas-berkas tersebut, lalu berlanjut memproses batch baru yang dikirim sesudahnya.

In [11]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Query file sink dimulai. Aktif:", query3.isActive)

Query file sink dimulai. Aktif: True


In [12]:
# Mengirim satu batch data baru
kirim_batch("batch_5.csv", [
    ["P016", "Wonosobo",   "Es Kopi Pandan", 2, 22000, fmt(sekarang + timedelta(minutes=6))],
    ["P017", "Temanggung", "Cold Brew",      1, 25000, fmt(sekarang + timedelta(minutes=6))],
])
print("Batch 5 dikirim. Menunggu Spark menulis ke Parquet...")
time.sleep(12)

Batch 5 dikirim. Menunggu Spark menulis ke Parquet...


In [13]:
# Bukti 1: berkas Parquet benar-benar tersimpan
!ls tugas10_output

part-00000-c088af78-2fb8-4df1-b1e1-7d217f9e6e5a-c000.snappy.parquet
part-00001-54311b3f-f3ee-4dea-9833-712759b32ee2-c000.snappy.parquet
part-00002-8c69eccd-716f-4851-9fec-5f13598642bb-c000.snappy.parquet
_spark_metadata


In [14]:
# Bukti 2: dibaca kembali sebagai DataFrame batch biasa
df_hasil = spark.read.parquet("tugas10_output")
print("isStreaming:", df_hasil.isStreaming)
df_hasil.orderBy("pesanan_id").show(truncate=False)

total_dikirim = sum(len(d) for d in semua_batch)
print(f"Jumlah baris di Parquet: {df_hasil.count()} | Jumlah baris yang dikirim: {total_dikirim}")

isStreaming: False
+----------+-----------+-----------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu        |qty|harga_satuan|waktu_pesan        |
+----------+-----------+-----------------+---+------------+-------------------+
|P001      |Magelang   |Es Kopi Gula Aren|3  |20000       |2026-11-06 12:46:08|
|P002      |Magelang   |Kopi Tubruk      |2  |12000       |2026-11-06 12:46:08|
|P003      |Purworejo  |Cold Brew        |1  |25000       |2026-11-06 12:46:08|
|P004      |Temanggung |Kopi Jahe        |4  |15000       |2026-11-06 12:46:08|
|P005      |Salatiga   |Flat White       |2  |24000       |2026-11-06 12:46:48|
|P006      |Magelang   |Es Kopi Pandan   |5  |22000       |2026-11-06 12:46:48|
|P007      |Purworejo  |Kopi Tubruk      |3  |12000       |2026-11-06 12:46:48|
|P008      |Wonosobo   |Kopi Jahe        |6  |15000       |2026-11-06 12:48:38|
|P009      |Temanggung |Es Kopi Gula Aren|2  |20000       |2026-11-06 12:48:38|
|P010      |Salatiga 

In [15]:
query_cabang.stop()
query_waktu.stop()
query3.stop()
print("Status aktif -> cabang:", query_cabang.isActive,
      "| waktu:", query_waktu.isActive, "| parquet:", query3.isActive)
print("Query aktif tersisa:", len(spark.streams.active))

spark.stop()
print("SparkSession ditutup.")

Status aktif -> cabang: False | waktu: False | parquet: False
Query aktif tersisa: 0
SparkSession ditutup.
